In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().resolve().parent.parent if str(Path.cwd()).endswith("notebooks") else Path.cwd().resolve()
FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
OUTPUT_PATH = ROOT / "work" / "outputs" / "capstone_metrics.json"

print("1. Loading the processed FlyRank feature data...")
features = pd.read_csv(FEATURE_PATH)
baseline = pd.read_csv(BASELINE_PATH)

numeric_cols = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

cat_cols = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

X_num = features[numeric_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
X_cat = features[cat_cols].fillna("unknown").astype(str)
X = pd.concat([
    X_num.reset_index(drop=True),
    pd.get_dummies(X_cat, prefix=cat_cols, dtype=float).reset_index(drop=True),
], axis=1)
y = features["is_declining_label"].astype(int)
groups = features["client_id"].astype(str)

print("2. Splitting by client to avoid leakage...")
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].reset_index(drop=True)
X_test = X.iloc[test_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

print("3. Training the logistic regression model...")
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])
model.fit(X_train, y_train)

test_df = features.iloc[test_idx].copy().reset_index(drop=True)
test_df["model_score"] = model.predict_proba(X_test)[:, 1]
base_map = baseline.set_index("content_id")["baseline_refresh_score"]
test_df["baseline_score"] = test_df["content_id"].map(base_map).fillna(0.0)


def precision_at_k(frame: pd.DataFrame, score_col: str, k: int) -> float:
    sorted_rows = frame.sort_values(score_col, ascending=False).head(k)
    return float(sorted_rows["is_declining_label"].mean()) if not sorted_rows.empty else 0.0


print("4. Computing the model-vs-baseline metrics...")
metrics = {
    "test_set_size": int(len(test_df)),
    "unique_clients": int(test_df["client_id"].nunique()),
    "base_rate": float(y_test.mean()),
    "baseline_p20": precision_at_k(test_df, "baseline_score", 20),
    "model_p20": precision_at_k(test_df, "model_score", 20),
    "baseline_p50": precision_at_k(test_df, "baseline_score", 50),
    "model_p50": precision_at_k(test_df, "model_score", 50),
    "baseline_p100": precision_at_k(test_df, "baseline_score", 100),
    "model_p100": precision_at_k(test_df, "model_score", 100),
    "baseline_auc": float(roc_auc_score(y_test, test_df["baseline_score"])),
    "model_auc": float(roc_auc_score(y_test, test_df["model_score"])),
    "baseline_ap": float(average_precision_score(y_test, test_df["baseline_score"])),
    "model_ap": float(average_precision_score(y_test, test_df["model_score"])),
}

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("\n--- DONE! Metrics written to work/outputs/capstone_metrics.json ---")
print(json.dumps(metrics, indent=2))


1. Loading dataset via DuckDB...


HTTPException: HTTP Error: HTTP GET error on 'hf://datasets/flyrank/flyrank-data/processed_signals.parquet' (HTTP 0 Internal Server Error)